# 03. 職業の知名度推定

「知らない職業に出会える」がこのアプリの核であり、これが計画段階から公開データが存在しない
唯一の部分だった。既存の統計・調査には「中学生が個々の職業をどれだけ知っているか」を
測ったものが無いため、代理指標を自分で作る。

## 検討した指標と却下したもの

- **検索ボリューム**: 無料で使える広告アカウント無しには取得できず、除外
- **教科書・学習指導要領での出現**: 著作物のスクレイピング/OCRが必要で権利的に難あり、除外
- **求人サイトの掲載数**: 労働市場の需要を測るものであり、子どもの認知度とは軸が違う。
  むしろ逆相関しうる（人手不足の職ほど求人は多いが子どもは知らない）。除外
- **Wikipedia日本語版**: 記事の有無・ページビューが公開APIで無料・低摩擦に取得できるため採用

## 2軸で持つ

記事の有無（「社会がその職業を一つのまとまりとして語っているか」）と、ページビュー
（「関心の強さ」）は性質が違う指標なので、1つのスコアに潰さず2軸で持つ。

## 指標の使い方の注意（重要）

**この知名度スコアは推薦の出し分け（知らない職業を優先的に見せる重み）にのみ使う。
職業の価値や優劣を表すものではない。** 知名度が低い＝劣った職業、では断じてない。


In [1]:
import re
import time

import numpy as np
import pandas as pd
import requests

from ipd_loader import load_description

pd.set_option("display.max_colwidth", 60)
pd.set_option("display.width", 200)

API = "https://ja.wikipedia.org/w/api.php"
PAGEVIEWS_API = "https://wikimedia.org/api/rest_v1/metrics/pageviews/per-article/ja.wikipedia/all-access/user/{article}/monthly/{start}/{end}"
HEADERS = {"User-Agent": "yumetane-research/0.1 (educational job-recommendation portfolio project)"}

desc, desc_labels = load_description()
names = desc[desc.columns[1]]
print("職業数:", len(names))


職業数: 556


## 職業名 → Wikipedia記事名の候補生成

job tagの職業名には、読点・括弧・スラッシュ・中黒などの区切り記号がある。これをどう
候補文字列に展開するかで照合率が決まるため、信頼度別にtierを付けて生成する。

- tier1: 職業名そのまま（最も信頼できる）
- tier2: 括弧の前の部分（例:「医薬情報担当者（MR）」→「医薬情報担当者」）
- tier3: スラッシュ区切りの各部分（例:「さく井工/ボーリング工」→両方）。job tagが
  明示的に別名として併記しているとみなせる
- tier4: 括弧の外にある読点区切りの各部分（例:「豆腐製造、豆腐職人」→両方）。同上
- tier5: 括弧の中身（例:「施設管理者（介護施設）」→「介護施設」）— **後述の理由で不採用**

中黒（・）は自動分割しない。「ハム・ソーセージ・ベーコン製造」のような複合語を機械的に
割ると意味を失う職業が多く、判別する簡単なルールが無いため。


In [2]:
def generate_candidates(name):
    tiers = [(1, name)]
    m = re.match(r"^(.*?)（(.+)）$", name)
    prefix, inside = None, None
    if m:
        prefix, inside = m.group(1).strip(), m.group(2).strip()
        if prefix and len(prefix) >= 2:
            tiers.append((2, prefix))
    if "/" in name or "／" in name:
        for p in re.split(r"[/／]", name):
            p = p.strip()
            if len(p) >= 2:
                tiers.append((3, p))
    base_for_comma = re.sub(r"（.*?）", "", name)
    if "、" in base_for_comma:
        for p in base_for_comma.split("、"):
            p = p.strip()
            if len(p) >= 2:
                tiers.append((4, p))
    if inside and "、" not in inside and len(inside) >= 2:
        tiers.append((5, inside))
    seen, result = set(), []
    for tier, cand in tiers:
        if cand not in seen:
            seen.add(cand)
            result.append((tier, cand))
    return result


job_candidates = {idx: generate_candidates(n) for idx, n in names.items()}
all_candidates = sorted({c for cands in job_candidates.values() for _, c in cands})
print("ユニーク候補文字列数:", len(all_candidates))


ユニーク候補文字列数: 696


## Wikipedia APIに一括照合する（レート制限に配慮してリトライ付き）

In [3]:
def batch(iterable, n):
    for i in range(0, len(iterable), n):
        yield iterable[i:i + n]


def query_titles(titles):
    params = {
        "action": "query", "titles": "|".join(titles), "redirects": 1,
        "prop": "pageprops", "format": "json", "formatversion": 2,
    }
    for attempt in range(6):
        r = requests.get(API, params=params, headers=HEADERS, timeout=15)
        if r.status_code == 429:
            wait = float(r.headers.get("Retry-After", 5)) * (attempt + 1)
            time.sleep(wait)
            continue
        r.raise_for_status()
        return r.json()
    raise RuntimeError("too many 429 retries")


resolved = {}
for chunk in batch(all_candidates, 50):
    data = query_titles(chunk)
    q = data.get("query", {})
    norm_map = {n["from"]: n["to"] for n in q.get("normalized", [])}
    redir_map = {rd["from"]: rd["to"] for rd in q.get("redirects", [])}
    pages = {p["title"]: p for p in q.get("pages", [])}
    for orig in chunk:
        t = redir_map.get(norm_map.get(orig, orig), norm_map.get(orig, orig))
        page = pages.get(t)
        if page is None or page.get("missing"):
            resolved[orig] = {"exists": False, "final_title": None, "disambig": False}
        else:
            disambig = "disambiguation" in page.get("pageprops", {})
            resolved[orig] = {"exists": True, "final_title": page["title"], "disambig": disambig}
    time.sleep(1.0)

print("照合した候補数:", len(resolved))


照合した候補数: 696


## tier5（括弧の中身）を全件目視した結果：不採用にする

自動照合の結果、tier5経由でヒットしたのは10件。全件を目視で確認したところ、**9件が
「職業記事ではなく一般的な概念・製品・業界の記事への誤爆」**だった。

| 職業名 | ヒットした記事 | 判定 |
|---|---|---|
| 施設管理者（介護施設） | 介護 | 誤爆（一般概念） |
| 食品営業（食品メーカー） | 日本の企業一覧(食料品) | 誤爆（無関係） |
| 代理店営業（保険会社） | 保険 | 誤爆（一般概念） |
| 検査工（工業製品） | 工業製品 | 誤爆（一般概念） |
| 自動運転開発エンジニア（自動車） | 自動車 | 誤爆（一般概念） |
| ホールスタッフ（レストラン） | レストラン | 誤爆（一般概念） |
| 商品企画開発（チェーンストア） | チェーンストア | 誤爆（一般概念） |
| 航空機開発エンジニア（ジェットエンジン） | ジェットエンジン | 誤爆（製品記事） |
| セキュリティエキスパート（デジタルフォレンジック） | デジタル・フォレンジック | 誤爆（分野記事） |
| 日本料理調理人（板前） | 板前 | 正しい |

括弧の中身は「別名」（医薬情報担当者（MR）、独立系ファイナンシャル・アドバイザー（IFA）
のような）と「限定・具体例」（施設管理者（介護施設）のような）の両方のパターンがあり、
機械的には区別できない。tier5は候補ソースとして不採用にする（自動照合には使わない）。

**job tag内部の「別名」列（`IPD_02_04_001`〜`025`）を追加候補にする案も検証したが、
効果は薄いと判断した。** 不一致だった職業の多くに別名はあるが、中身を見ると「乳製品製造」
→「牛乳製造工」「チーズ製造工」のように、**別名は元の職業名よりさらに専門的・細分化された
呼称**であり、Wikipediaに単独記事がある可能性はむしろ低い。例外的に「清酒製造」→「杜氏」
のように広く知られた別称を拾えるケースもあるが、稀だった。


In [4]:
def best_match(idx):
    cands = job_candidates[idx]
    for tier, cand in cands:
        if tier == 5:
            continue  # tier5は不採用
        info = resolved[cand]
        if info["exists"] and not info["disambig"]:
            return tier, cand, info["final_title"], "matched"
    for tier, cand in cands:
        if tier == 5:
            continue
        info = resolved[cand]
        if info["exists"] and info["disambig"]:
            return tier, cand, None, "disambig"
    return None, None, None, "unmatched"


rows = []
for idx, name in names.items():
    tier, cand, title, status = best_match(idx)
    rows.append({
        "収録番号": idx, "職業名": name, "tier": tier,
        "candidate": cand, "wiki_title": title, "status": status,
    })

match_df = pd.DataFrame(rows).set_index("収録番号")
match_df["status"].value_counts()


status
unmatched    333
matched      217
disambig       6
Name: count, dtype: int64

## 板前だけはtier5でも例外的に採用する

上の関数はtier5を機械的に除外しているため、「日本料理調理人（板前）」も現時点では
unmatchedになっている。これは目視確認で「板前」が正しい記事だと確認済みなので、
1件だけ手で復帰させる。


In [5]:
itamae_idx = names[names == "日本料理調理人（板前）"].index[0]
match_df.loc[itamae_idx, ["tier", "candidate", "wiki_title", "status"]] = [5, "板前", "板前", "matched"]
match_df.loc[itamae_idx]


職業名           日本料理調理人（板前）
tier                  5.0
candidate              板前
wiki_title             板前
status            matched
Name: 98, dtype: object

## 曖昧さ回避12件を手動で解決する

自動照合で「記事はあるが曖昧さ回避ページ」だったのは12件。各ページのリンク先と、
job tagの職業解説文（`IPD_03_01_000`）を突き合わせて、正しい語義の記事を手動で特定した。

| 職業名 | 由来tier | 解決結果 |
|---|---|---|
| 西洋料理調理人（コック） | 5 | → コック (職業)（解決） |
| キャディ | 1 | → キャディ (ゴルフ)（解決） |
| パイロット | 1 | → パイロット (航空)（解決） |
| 独立系ファイナンシャル・アドバイザー（IFA） | 5 | → 独立系ファイナンシャルアドバイザー（解決） |
| トリマー | 1 | ペット関連の単独記事なし → unmatched |
| フロント（ホテル・旅館） | 2 | ホテルフロント業務の単独記事なし → unmatched |
| ディーラー | 1 | job tagの語義（証券・銀行の自己資金売買）の記事なし → unmatched |
| UX/UIデザイナー | 3 | UI/UXデザイナーの単独記事なし → unmatched |
| 運用・管理（IT） | 5 | 該当職務の単独記事なし → unmatched |
| セキュリティエキスパート（オペレーション） | 5 | 該当職務の単独記事なし → unmatched |
| コンサルティング営業（IT） | 5 | 該当職務の単独記事なし → unmatched |
| オーケストラ奏者（団員） | 5 | 演奏家個人としての単独記事なし → unmatched |

トリマー・フロント・ディーラー・UX/UIデザイナーは、tier1〜3という比較的信頼できる候補
から来ていたにもかかわらず、正しい語義の記事自体が存在しなかった。これは照合の失敗では
なく、Wikipedia側にその職業の記事が無いという事実である。


In [6]:
disambig_resolution = {
    "西洋料理調理人（コック）": "コック (職業)",
    "キャディ": "キャディ (ゴルフ)",
    "パイロット": "パイロット (航空)",
    "独立系ファイナンシャル・アドバイザー（IFA）": "独立系ファイナンシャルアドバイザー",
}

for job_name, title in disambig_resolution.items():
    idx = names[names == job_name].index[0]
    match_df.loc[idx, ["wiki_title", "status"]] = [title, "matched"]

# 残りのdisambig（記事なしと判定したもの）はunmatchedに落とす
remaining_disambig_names = [
    "トリマー", "フロント（ホテル・旅館）", "ディーラー", "UX/UIデザイナー",
    "運用・管理（IT）", "セキュリティエキスパート（オペレーション）",
    "コンサルティング営業（IT）", "オーケストラ奏者（団員）",
]
for job_name in remaining_disambig_names:
    idx = names[names == job_name].index[0]
    match_df.loc[idx, ["wiki_title", "status"]] = [None, "unmatched"]

match_df["has_article"] = match_df["status"] == "matched"
match_df["status"].value_counts()


status
unmatched    334
matched      222
Name: count, dtype: int64

In [7]:
n = len(match_df)
matched = match_df["has_article"].sum()
print(f"記事あり(has_article=True): {matched}/{n} = {matched/n:.1%}")
print(f"記事なし(has_article=False): {n - matched}/{n} = {(n-matched)/n:.1%}")


記事あり(has_article=True): 222/556 = 39.9%
記事なし(has_article=False): 334/556 = 60.1%


## ページビュー（副指標）：記事がある職業だけ、直近12か月の月次中央値

記事の有無は556件全件をカバーする主指標。ページビューは記事がある職業の中での
関心の強弱を見る副指標として、直近12か月の月次中央値を使う（単月だとドラマ化などの
一時的なスパイクに引っ張られるため）。


In [8]:
def get_monthly_pageviews(article, start="20250801", end="20260801"):
    url = PAGEVIEWS_API.format(article=requests.utils.quote(article, safe=""), start=start, end=end)
    for attempt in range(6):
        r = requests.get(url, headers=HEADERS, timeout=15)
        if r.status_code == 429:
            time.sleep(5 * (attempt + 1))
            continue
        if r.status_code == 404:
            return []
        r.raise_for_status()
        return [item["views"] for item in r.json().get("items", [])]
    raise RuntimeError("too many 429 retries")


pageview_median = {}
matched_idx = match_df.index[match_df["has_article"]]
for i, idx in enumerate(matched_idx):
    title = match_df.loc[idx, "wiki_title"]
    views = get_monthly_pageviews(title)
    pageview_median[idx] = float(np.median(views)) if views else np.nan
    if (i + 1) % 40 == 0:
        print(f"{i + 1}/{len(matched_idx)} 件取得済み")
    time.sleep(0.3)

match_df["pageviews_median_12mo"] = pd.Series(pageview_median)
match_df["pageviews_median_12mo"].describe()


40/222 件取得済み


80/222 件取得済み


120/222 件取得済み


160/222 件取得済み


200/222 件取得済み


count      221.000000
mean      1079.567873
std       1370.011652
min          3.500000
25%        280.000000
50%        748.000000
75%       1286.000000
max      13653.000000
Name: pageviews_median_12mo, dtype: float64

## 検証：「記事が無い＝子どもが知らない」は本当か

ここまでの指標は「記事があるか」を主軸に置いているが、これは検証されていない仮説である。
556職業から50件を無作為抽出し、**中学生が名前を聞いて仕事内容を説明できそうか**という
基準で、自分の判断で3段階のラベルを付けた（知っている／名前は聞いたことがある／知らない）。

**このラベル付けの限界について正直に書いておく。** これは実際の生徒を対象にした調査では
なく、一般的な日本の生活文化・メディア露出についての常識的な推測に基づく、あくまで
一人の判断による代理ラベルである。本来は複数人でのアノテーションや実際の生徒への
聞き取りで裏取りすべきものであり、n=50・単一評価者という制約は指標の限界として明記する。
それでも「自分で作った指標を、自分で検証しようとした」という工程自体に意味があるという
判断で実施する。


In [9]:
sample_idx = names.sample(n=50, random_state=42).sort_index().index
sample_df = pd.DataFrame({"職業名": names.loc[sample_idx]})
sample_df


,職業名
2,洋菓子製造、パティシエ
6,冷凍加工食品製造
10,しょうゆ製造
30,建設機械オペレーター
55,スーパー店員
70,ペットショップ店員
73,フランチャイズチェーン・スーパーバイザー
76,コンビニエンスストア店員
78,中小企業診断士
81,社会保険労務士


判断基準:

- **知っている**: 職業名を見て、仕事内容を自分の言葉である程度説明できそうな、日常的に
  接触機会がある職業（コンビニ店員、パティシエ、国会議員など）
- **名前は聞いたことがある**: 単語自体はメディアや周囲で見聞きしたことがありそうだが、
  具体的な仕事内容までは説明できなさそうな職業
- **知らない**: 職業名を見ても、それが何をする仕事か見当がつかなさそうな職業（専門資格・
  業界特化の職種区分など）


In [10]:
manual_labels = {
    "洋菓子製造、パティシエ": "知っている",
    "冷凍加工食品製造": "知らない",
    "しょうゆ製造": "名前は聞いたことがある",
    "建設機械オペレーター": "名前は聞いたことがある",
    "スーパー店員": "知っている",
    "ペットショップ店員": "知っている",
    "フランチャイズチェーン・スーパーバイザー": "知らない",
    "コンビニエンスストア店員": "知っている",
    "中小企業診断士": "知らない",
    "社会保険労務士": "知らない",
    "土地家屋調査士": "知らない",
    "通訳者": "知っている",
    "訪問介護員/ホームヘルパー": "名前は聞いたことがある",
    "麻薬取締官": "名前は聞いたことがある",
    "家庭裁判所調査官": "知らない",
    "助産師": "知っている",
    "保健師": "名前は聞いたことがある",
    "歯科技工士": "知らない",
    "理学療法士（PT）": "名前は聞いたことがある",
    "言語聴覚士": "知らない",
    "通関士": "知らない",
    "テレビ・ラジオ放送技術者": "名前は聞いたことがある",
    "植物工場の設計、施工": "知らない",
    "家電修理": "知っている",
    "織布工/織機オペレーター": "知らない",
    "靴製造": "名前は聞いたことがある",
    "ソフトウェア開発（スマホアプリ）": "知っている",
    "パタンナー": "知らない",
    "スタイリスト": "知っている",
    "職業訓練指導員": "知らない",
    "小児科医": "知っている",
    "調香師": "名前は聞いたことがある",
    "データ入力": "知っている",
    "経理事務": "名前は聞いたことがある",
    "営業事務": "知らない",
    "銀行・信用金庫渉外担当": "知らない",
    "清涼飲料ルートセールス": "知らない",
    "雑踏・交通誘導警備員": "知っている",
    "ピッキング作業員": "知らない",
    "バックヤード作業員（スーパー食品部門）": "知らない",
    "港湾荷役作業員": "知らない",
    "国会議員": "知っている",
    "入国審査官": "名前は聞いたことがある",
    "労働基準監督官": "知らない",
    "郵便局郵便窓口業務": "知っている",
    "障害者グループホーム世話人": "知らない",
    "自動車板金塗装": "名前は聞いたことがある",
    "オーケストラ奏者（団員）": "知っている",
    "税関職員": "名前は聞いたことがある",
    "障害福祉サービス管理責任者": "知らない",
}

sample_df["awareness_label"] = sample_df["職業名"].map(manual_labels)
assert sample_df["awareness_label"].isna().sum() == 0, "ラベル未設定の職業がある"

label_order = ["知らない", "名前は聞いたことがある", "知っている"]
sample_df["awareness_score"] = sample_df["awareness_label"].map(
    {l: i for i, l in enumerate(label_order)}
)
sample_df["has_article"] = match_df.loc[sample_idx, "has_article"]
sample_df["pageviews_median_12mo"] = match_df.loc[sample_idx, "pageviews_median_12mo"]
sample_df


,職業名,awareness_label,awareness_score,has_article,pageviews_median_12mo
2,洋菓子製造、パティシエ,知っている,2,True,1651.0
6,冷凍加工食品製造,知らない,0,False,NaN
10,しょうゆ製造,名前は聞いたことがある,1,False,NaN
30,建設機械オペレーター,名前は聞いたことがある,1,False,NaN
55,スーパー店員,知っている,2,False,NaN
70,ペットショップ店員,知っている,2,False,NaN
73,フランチャイズチェーン・スーパーバイザー,知らない,0,False,NaN
76,コンビニエンスストア店員,知っている,2,False,NaN
78,中小企業診断士,知らない,0,True,2492.0
81,社会保険労務士,知らない,0,True,3245.0


## 記事有無との一致度

In [11]:
cross = pd.crosstab(sample_df["awareness_label"], sample_df["has_article"])
cross = cross.reindex(label_order)
cross


has_article,False,True
awareness_label,,
知らない,11,11
名前は聞いたことがある,6,7
知っている,8,7


In [12]:
# 「知っている」を認知度が高い、「知らない」を低いとみなしたときの単純な整合率
# has_article=True が「知っている」または「名前は聞いたことがある」に対応していれば妥当、とみなす
sample_df["has_article_expected_known"] = sample_df["awareness_label"] != "知らない"
agreement = (sample_df["has_article"] == sample_df["has_article_expected_known"]).mean()
print(f"「記事あり ⇔ 知っている/聞いたことがある」の一致率: {agreement:.1%}")

# ズレの内訳
mismatch = sample_df[sample_df["has_article"] != sample_df["has_article_expected_known"]]
mismatch[["職業名", "awareness_label", "has_article", "pageviews_median_12mo"]]


「記事あり ⇔ 知っている/聞いたことがある」の一致率: 50.0%


,職業名,awareness_label,has_article,pageviews_median_12mo
10,しょうゆ製造,名前は聞いたことがある,False,NaN
30,建設機械オペレーター,名前は聞いたことがある,False,NaN
55,スーパー店員,知っている,False,NaN
70,ペットショップ店員,知っている,False,NaN
76,コンビニエンスストア店員,知っている,False,NaN
78,中小企業診断士,知らない,True,2492.0
81,社会保険労務士,知らない,True,3245.0
84,土地家屋調査士,知らない,True,1724.0
144,家庭裁判所調査官,知らない,True,258.0
163,歯科技工士,知らない,True,418.0


## ページビューとの関係（記事がある職業のみ）

In [13]:
with_article = sample_df[sample_df["has_article"]]
with_article.groupby("awareness_label")["pageviews_median_12mo"].agg(["count", "median", "mean"]).reindex(label_order)


,count,median,mean
awareness_label,,,
知らない,11,934.0,1160.545455
名前は聞いたことがある,7,758.0,1303.285714
知っている,7,749.0,1035.428571


## 結果の読み方（n=50・単一評価者という前提つき）

- 一致率と、ズレの中身は上のセルの実行結果を見て判断する
- ズレが「知っている」なのに記事が無いケースに集中していれば、記事有無は再現率
  （知られている職業を取りこぼす方向）に弱いということになる
- 逆に「知らない」なのに記事があるケースが多ければ、Wikipediaは一般層・専門コミュニティの
  関心を反映しており、中学生の認知度とは軸がずれている可能性がある
- ページビューが「知っている」で高く「知らない」で低い傾向があれば、副指標としての妥当性の
  補強材料になる
